In [1]:
%pylab inline

import numpy as np
import numpy.ma as ma

import scipy as sp
import scipy.signal as sg
import scipy.ndimage as ndimage

import importlib

import matplotlib
import matplotlib.pyplot as plt

import sys, os
# sys.path.append('/home/tarasenko/prj/posada3/') #last version on my computer
# import wind_func as wnd

sys.path.append('/home/tarasenko/codes/python/data_analysis/')
from data_analysis import simplest_1d_filter

sys.path.append('/home/tarasenko/codes/python/wind/')

import s1_wind_list as wlist
from s1_wind_list import pn, pol, resolution, distance, area_x, area_y, xs, ys

import s1_with_posada as s1_pos
importlib.reload(s1_pos)

import wind_func as wnd
importlib.reload(wnd)

sys.path.append('/home/tarasenko/codes/python/satdata')
import geotiff_open as gtf
import spatial_data_operations as spat
importlib.reload(gtf)
importlib.reload(spat)

sys.path.append('/home/tarasenko/codes/python/Sentinel_1/')
from draft_frommeanaverage import return_bin_icemask


res_m = 500

# fpath_ice = '/home/tarasenko/data/ice_mask/'

Populating the interactive namespace from numpy and matplotlib


In [17]:
# 1) Import sigma and correct thermal noise
importlib.reload(s1_pos)
importlib.reload(wnd)


# fn = wlist.fn_list[0]
# test 1: FN to check it with ice mask (known filename)
# fn ='S1A_EW_GRDM_1SDH_20160511T023241_20160511T023346_011205_010EF1_A86C.zip'

# test 2: check how to work with dictionaries
wind_cond = 'on'

on_wind_data_global=[]
on_wind_distances_global=[]
off_wind_data_global=[]
off_wind_distances_global=[]


fn_list = {}
fn_list_off['S1A_EW_GRDM_1SDH_20170312T024142_20170312T024242_015653_019BFD_1FD1.zip']={"area_x" = [400, 800],
                                                                                        "area_y" = [0, 400]}

SyntaxError: invalid syntax (<ipython-input-17-fcd157adadea>, line 20)

In [ ]:
for fn in fn_list.keys():
    print (fn)
    s0, lat, lon, inc, sigma0_corr, s0_nice, wdir_model = s1_pos.get_data(pn+fn, res_m = res_m, pol = pol, wind_model = 'interim')


    # 2) Find an ice mask if there is one. The ice mask were prepared by Eduard Kazakov in Nansen Center in StP in 2017, aug
    # the ice_mask is a geotiff file with a spatial resolution of initial S1 image (40 m). We take only every 10th point 
    # to reduce the size of the matrix (cause my computer is too slow here, in StP)

    masked_s0_corr =[]
    for f_ice_mask in os.listdir('/home/tarasenko/data/ice_mask/'):
        if f_ice_mask[4:] == fn[:-4]:
            xsize, ysize, GT, geoproj, band_400 = gtf.readFile(fpath_ice+f_ice_mask+'/icewater.tif', nth_point=10)
            lon_rastr, lat_rastr = gtf.get_latlon(band_data, GT, nth_point=10)
    #         band_400 = band_data.squeeze()[::10,::10]
            masked_s0_corr = spat.create_mask(band_400, lat_rastr, lon_rastr,\
                                              sigma0_corr, lat, lon, condition = lambda x: x==2)

    # Zaschita ot duraka: na sluchai if there is no ice mask available     
    if masked_s0_corr == []:
#         masked_s0_corr = sigma0_corr
        bin_ice = return_bin_icemask(pn, fn, res_m)
        masked_s0_corr = np.ma.masked_where(bin_ice==0, s0)


    # 1.1. Plot 
    fig_sigma = plt.figure(figsize=(12, 12))
    plt.subplot(2, 2, 1)
    plt.pcolormesh(10*np.log10(s0), cmap='jet')
    plt.clim(-40, -5)
    plt.colorbar()
    plt.title(fn)

    plt.subplot(2, 2, 2)
    plt.pcolormesh(10*np.log10(sigma0_corr), cmap='jet')
    plt.clim(-40, -5)
    plt.colorbar()
    plt.title('s_0 - thermal noise')


    plt.subplot(2, 2, 3)
    plt.pcolormesh(10*np.log10(masked_s0_corr), cmap='jet')
    plt.clim(-40, -5)
    plt.colorbar()
    plt.title('sigma_0 corrected with ice mask')


    plt.subplot(2, 2, 4)
    plt.pcolormesh(s0_nice, cmap='jet')
    plt.clim(-5, 5)
    plt.colorbar()
    plt.title('nice image')
    plt.show()


    figname = '/home/tarasenko/Figures/tests_wind/'+fn[:-4]+'_sigma.png'
    fig_sigma.savefig(figname, dpi = 300)

In [ ]:
# 2. Wind: load wind calculated from raw sigma and from corrected
# importlib.reload(s1_pos)


wind_speed = s1_pos.get_wind_data(pn+fn, s0, inc, wdir_model, lat, lon)

wind_speed_corr = s1_pos.get_wind_data(pn+fn, masked_s0_corr, inc, wdir_model, lat, lon)


# 2.1. Plot 
fig_wind = plt.figure(figsize=(20, 10))
plt.subplot(1, 2, 1)
wnd.wind_plot(fn, wind_speed, wdir_model, lat, lon, every_x = 50, case = 'ERA INTERIM')

plt.subplot(1, 2, 2)
wnd.wind_plot(fn, wind_speed_corr, wdir_model, lat, lon, every_x = 50, case = 'ERA INTERIM')
plt.show()

plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plt.pcolormesh(s0 - sigma0_corr, cmap='jet')
plt.axis('image')
plt.colorbar()
plt.title('thermal noise')


plt.subplot(1, 2, 2)
plt.pcolormesh(wind_speed - wind_speed_corr, cmap='jet')
plt.axis('image')
plt.colorbar()
plt.clim(-1, 1)

plt.title('wind_speed - wind_speed_corr, m/s')
plt.show()


figname = '/home/tarasenko/Figures/tests_wind/'+fn[:-4]+'_wind-test.png'
fig_wind.savefig(figname, dpi = 300)

In [ ]:

xs = fn_list[fn]['xs']
ys = fn_list[fn]['ys']
try:
    distance = fn_list[fn]['distance']
except:
    distance = 5e4


try:
    area_x = fn_list[fn]['area_x']
    area_y = fn_list[fn]['area_y']
    
    s0_area = s0[area_x[0]:area_x[1], area_y[0]:area_y[1]]
    sigma0_corr_area = sigma0_corr[area_x[0]:area_x[1], area_y[0]:area_y[1]]
    lat_area = lat[area_x[0]:area_x[1], area_y[0]:area_y[1]]
    lon_area = lon[area_x[0]:area_x[1], area_y[0]:area_y[1]]
    wind_speed_area = wind_speed[area_x[0]:area_x[1], area_y[0]:area_y[1]]
    wind_speed_corr_area = wind_speed_corr[area_x[0]:area_x[1], area_y[0]:area_y[1]]
    wdir_model_area = wdir_model[area_x[0]:area_x[1], area_y[0]:area_y[1]]
except:
    s0_area = s0
    sigma0_corr_area = sigma0_corr
    lat_area = lat
    lon_area = lon
    wind_speed_area = wind_speed
    wind_speed_corr_area = wind_speed_corr
    wdir_model_area = wdir_model
    
    
# s0_temp.shape   
    

from scipy import ndimage, misc
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(10, 5))

# ax1 = fig.add_subplot(121)  # left side
# ax2 = fig.add_subplot(122)  # right side
plt.subplot(121)
result = ndimage.gaussian_filter(wind_speed_area,  sigma=2)
plt.pcolormesh(wind_speed_area, cmap = 'jet')
plt.axis('image')

plt.subplot(122)
plt.pcolormesh(result, cmap = 'jet')
plt.axis('image')
# quadmesh.set_clim(vmin=0, vmax=15)
# plt.clim([-10, 10])

plt.show()

importlib.reload(wnd)
# Plotting the sections along the mean wind direction

# from s1_wind_list import pn, pol, resolution, distance, area_x, area_y, xs, ys
xs = fn_list[fn]['xs']
ys = fn_list[fn]['ys']
print(xs)


# ys = [360, 320, 300]
# xs = [50, 150, 250]

window_size = 5    # cut of the highest frequencies corresponding to x points 
window_size_km = window_size*resolution/1000 #optional - need for plotting 


#try to extract multiple sections at different places


ccc = ['black', 'yellow', 'magenta', 'red']
section_coords = {'x0':[], 'x1':[], 'y0':[], 'y1':[]}
distance_vector = {}
data_section = {}
polynoms = {}

n = 0
for x0, y0 in zip(xs, ys):
#         print (x0, y0)
        data_section[n], distance_vector[n], section_coords_temp = wnd.section_extract(wind_speed_corr_area, x0, y0, \
                                                                                   np.mean(wdir_model_area.squeeze()),\
                                                                                   distance, resolution)
        section_coords['x0'].append(section_coords_temp[0])
        section_coords['x1'].append(section_coords_temp[1])
        section_coords['y0'].append(section_coords_temp[2])
        section_coords['y1'].append(section_coords_temp[3])
        n+=1

# Mask out "high winds" ~ sea ice
if 'no filter' in fn_list[fn]:
    pass    
else: 
    data_section = simplest_1d_filter(data_section, n_std = 3)



# Plotplotplot

# fig_sections = plt.figure(figsize=(18, 30))
# plt.subplot(3, 1, 1)
fig_sections_area = plt.figure(figsize=(10, 10))
wnd.wind_plot(fn, wind_speed_corr_area, wdir_model_area,
              lat_area, lon_area,
              every_x = 50, case = 'ERA INTERIM',
              section_coords=section_coords, sect_cols=ccc)
plt.axis('image')
plt.show()




# plt.subplot(3, 1, 2, axisbg='lightgrey')
fig_sections = plt.figure(figsize=(15, 6))
plt.subplot(1, 1, 1, axisbg='lightgrey')

for n in np.arange(len(section_coords['x0'])):
    distance_km = distance_vector[n]/1000 
    plt.scatter(distance_km, data_section[n], label = 'section '+str(n),  color = ccc[n], linewidth = 1.5)
    
    deg = 2
    z = np.ma.polyfit(distance_km, data_section[n], deg)
    polynoms[n] = np.poly1d(z) #polynomial structure (special datatype in python)
#     plt.plot(distance_km, p(distance_km), label = str(z[0])+'x +'+str(z[1]))    
    plt.plot(distance_km, polynoms[n](distance_km), label = str(polynoms[n]), color = ccc[n], linewidth = 3.5)    
    
plt.legend()   
plt.title("Wind speed along sections and their regression of "+str(deg)+'deg')   
plt.xlabel('distance, km')
plt.ylabel('wind speed, m/s')



# plt.subplot(3, 1, 3, axisbg='lightgrey')
# for n in np.arange(len(section_coords['x0'])):
#     N  = 5    # Filter order
#     Wn = 1/window_size   #cut-off _spatial_ frequency (cut the highest frequencies corresponding to n_points)
#     B, A = sg.butter(N, Wn, output='ba')

#     # Second, apply the filter
#     data_f = sg.filtfilt(B, A, data_section[n][data_section[n]<18])
#     plt.plot(distance_km[data_section[n]<18], data_f, label = 'section '+str(n), color = ccc[n])   
# plt.title("Butterworth filter, window size "+str(window_size_km)+' km')
# plt.xlabel('distance, km')
# plt.ylabel('wind speed, m/s')



plt.tight_layout()
plt.show()

if area_x!=[] and area_y!=[]:
    figname = '/home/tarasenko/Figures/tests_wind/'+fn[:-4]+'-'+str(area_x[0])\
    +'-'+str(area_x[1])+'-'+str(area_y[0])+'-'+str(area_y[1])
else:
    figname = '/home/tarasenko/Figures/tests_wind/'+fn[:-4]

fig_sections_area.savefig(figname+'_wind_sections_area.png', dpi = 300)    
fig_sections.savefig(figname+'_wind_sections.png', dpi = 300)

In [ ]:
# Plotting the bunch of sections along the mean wind direction
# print (xs[0], xs[-1], xs)
# print (ys[0], ys[-1], ys)

distance = 10e4
# xs = [250, 270, 360]
# ys = [220, 260, 300]

window_size = 5    # cut-off the highest frequencies corresponding to x points 
window_size_km = window_size*resolution/1000 #optional - need for plotting 


#try to extract multiple sections at different places
xs_full = np.arange(xs[0], xs[-1], 10)
if ys[0]!= ys[-1]:
    ys_full = np.linspace(ys[0], ys[-1], len(xs_full))
else:
    ys_full = np.repeat(ys[0], len(xs_full))
    

ccc = ['black', 'yellow', 'magenta', 'red']
section_coords = {'x0':[], 'x1':[], 'y0':[], 'y1':[]}
distance_vector = {}
data_section = {}
polynoms = {}
approximation = []

n = 0
for x0, y0 in zip(xs_full, ys_full):
        data_section[n], distance_vector[n], section_coords_temp = wnd.section_extract(wind_speed_corr_area, x0, y0, \
                                                                                   np.mean(wdir_model_area.squeeze()),\
                                                                                   distance, resolution)
        section_coords['x0'].append(section_coords_temp[0])
        section_coords['x1'].append(section_coords_temp[1])
        section_coords['y0'].append(section_coords_temp[2])
        section_coords['y1'].append(section_coords_temp[3])
        n+=1

        
# Mask out "high winds" ~ sea ice
if 'no filter' in fn_list[fn]:
    pass    
else: 
    data_section = simplest_1d_filter(data_section, n_std = 3)
    
    
# Append to global database
for n in np.arange(len(section_coords['x0'])):    
    if wind_cond == 'on':
        on_wind_data_global.append(data_section[n])
        on_wind_distances_global.append(distance_km)

    if wind_cond == 'off':
        off_wind_data_global.append(data_section[n])
        off_wind_distances_global.append(distance_km)  


        
# Plotplotplot

# fig_sections = plt.figure(figsize=(18, 30))
# plt.subplot(3, 1, 1)
fig_sections_area = plt.figure(figsize=(8, 8))
wnd.wind_plot(fn, wind_speed_corr_area, wdir_model_area,
              lat_area, lon_area,
              every_x = 50, case = 'ERA INTERIM',
              section_coords=section_coords, sect_cols = 'black')

plt.axis('image')
plt.show()




# plt.subplot(3, 1, 2, axisbg='lightgrey')
fig_sections = plt.figure(figsize=(15, 6))
plt.subplot(1, 1, 1, axisbg='lightgrey')
for n in np.arange(len(section_coords['x0'])):
    distance_km = distance_vector[n]/1000 
    plt.plot(distance_km, data_section[n], 'o',  color = 'black', linewidth = 0.75)
    
    
    deg = 4
    z = np.ma.polyfit(distance_km, data_section[n], deg)
    polynoms[n] = np.poly1d(z) #polynomial structure (special datatype in python)
    approximation.append(polynoms[n](distance_km))
      
        
plt.plot(distance_km, np.mean(approximation, 0), color = 'red', linewidth = 3)     
plt.title("Wind speed along "+str(n+1)+" sections and the mean of it's approximations (of "+str(deg)+" degree)")   
plt.xlabel('distance, km')
plt.ylabel('wind speed, m/s')


plt.tight_layout()
plt.show()

if area_x!=[] and area_y!=[]:
    figname = '/home/tarasenko/Figures/tests_wind/'+fn[:-4]+'-'+str(area_x[0])\
    +'-'+str(area_x[1])+'-'+str(area_y[0])+'-'+str(area_y[1])
else:
    figname = '/home/tarasenko/Figures/tests_wind/'+fn[:-4]

fig_sections_area.savefig(figname+'_wind_sections_area_every10km.png', dpi = 300)    
fig_sections.savefig(figname+'_wind_sections_every10km.png', dpi = 300)

In [ ]:
# importlib.reload(wnd)


# # Plotting the sections along the mean wind direction ising the smoothed data


# distance = 10e4


# window_size = 5    # cut of the highest frequencies corresponding to x points 
# window_size_km = window_size*resolution/1000 #optional - need for plotting 


# #try to extract multiple sections at different places


# ccc = ['black', 'yellow', 'magenta', 'red']
# section_coords = {'x0':[], 'x1':[], 'y0':[], 'y1':[]}
# distance_vector = {}
# data_section = {}
# polynoms = {}

# n = 0
# for x0, y0 in zip(xs, ys):
# #         print (x0, y0)
#         data_section[n], distance_vector[n], section_coords_temp = wnd.section_extract(result, x0, y0, \
#                                                                                    np.mean(wdir_model_area.squeeze()),\
#                                                                                    distance, resolution)
#         section_coords['x0'].append(section_coords_temp[0])
#         section_coords['x1'].append(section_coords_temp[1])
#         section_coords['y0'].append(section_coords_temp[2])
#         section_coords['y1'].append(section_coords_temp[3])
#         n+=1


        
# # Mask out "high winds" ~ sea ice
# if 'no filter' in fn_list[fn]:
#     pass    
# else: 
#     data_section = simplest_1d_filter(data_section, n_std = 3)




# # Plotplotplot

# # fig_sections = plt.figure(figsize=(18, 30))
# # plt.subplot(3, 1, 1)
# fig_sections_area = plt.figure(figsize=(10, 10))
# wnd.wind_plot(fn, result, wdir_model_area,
#               lat_area, lon_area,
#               every_x = 50, case = 'ERA INTERIM',
#               section_coords=section_coords, sect_cols=ccc)
# plt.clim([0, 20])
# plt.axis('image')
# plt.show()




# # plt.subplot(3, 1, 2, axisbg='lightgrey')
# fig_sections = plt.figure(figsize=(15, 6))
# plt.subplot(1, 1, 1, axisbg='lightgrey')

# for n in np.arange(len(section_coords['x0'])):
# #     print(n)   
#     distance_km = distance_vector[n]/1000 
#     plt.scatter(distance_km, data_section[n], label = 'section '+str(n),  color = ccc[n], linewidth = 1.5)
    
    
#     deg = 2
#     z = np.ma.polyfit(distance_km, data_section[n], deg)
#     polynoms[n] = np.poly1d(z) #polynomial structure (special datatype in python)
# #     plt.plot(distance_km, p(distance_km), label = str(z[0])+'x +'+str(z[1]))    
#     plt.plot(distance_km, polynoms[n](distance_km), label = str(polynoms[n]), color = ccc[n], linewidth = 3.5)    

    
# plt.legend()

    
# plt.title("Wind speed along sections and their regression of "+str(deg)+'deg')   
# plt.xlabel('distance, km')
# plt.ylabel('wind speed, m/s')


# # #Butterworth filter
# # plt.subplot(3, 1, 3, axisbg='lightgrey')
# # for n in np.arange(len(section_coords['x0'])):
# #     N  = 5    # Filter order
# #     Wn = 1/window_size   #cut-off _spatial_ frequency (cut the highest frequencies corresponding to n_points)
# #     B, A = sg.butter(N, Wn, output='ba')

# #     # Second, apply the filter
# #     data_f = sg.filtfilt(B, A, data_section[n][data_section[n]<18])
# #     plt.plot(distance_km[data_section[n]<18], data_f, label = 'section '+str(n), color = ccc[n])
# # plt.title("Butterworth filter, window size "+str(window_size_km)+' km')
# # plt.xlabel('distance, km')
# # plt.ylabel('wind speed, m/s')



# plt.tight_layout()
# plt.show()

# if area_x!=[] and area_y!=[]:
#     figname = '/home/tarasenko/Figures/tests_wind/'+fn[:-4]+'-'+str(area_x[0])\
#     +'-'+str(area_x[1])+'-'+str(area_y[0])+'-'+str(area_y[1])
# else:
#     figname = '/home/tarasenko/Figures/tests_wind/'+fn[:-4]

# fig_sections_area.savefig(figname+'_wind_sections_area.png', dpi = 300)    
# fig_sections.savefig(figname+'_wind_sections.png', dpi = 300)

In [ ]:
# fig_sections = plt.figure(figsize=(18, 30))
# plt.subplot(3, 1, 1)
# wnd.wind_plot(fn, wind_speed_corr_area, wdir_model_area,
#               lat_area, lon_area,
#               every_x = 50, case = 'ERA INTERIM',
#               section_coords=section_coords, sect_cols=ccc)

# plt.axis('image')
# plt.show()


# plt.subplot(3, 1, 2, axisbg='lightgrey')
# for n in np.arange(len(section_coords['x0'])):
#     distance_km = distance_vector[n]/1000 
#     plt.plot(distance_km, data_section[n], label = 'section '+str(n),  color = ccc[n], linewidth = 1.5)
    
    
#     deg = 4
#     z = np.polyfit(distance_km, data_section[n], deg)
#     polynoms[n] = np.poly1d(z) #polynomial structure (special datatype in python)
#     plt.plot(distance_km, polynoms[n](distance_km), label = str(polynoms[n]), color = ccc[n], linewidth = 3.5)      
# plt.legend()

    
# plt.title("Wind speed along sections")   
# plt.xlabel('distance, km')
# plt.ylabel('wind speed, m/s')


# plt.subplot(3, 1, 3, axisbg='lightgrey')
# for n in np.arange(len(section_coords['x0'])):
#     N  = 5    # Filter order
#     Wn = 1/window_size   #cut-off _spatial_ frequency (cut the highest frequencies corresponding to n_points)
#     B, A = sg.butter(N, Wn, output='ba')

#     # Second, apply the filter
#     data_f = sg.filtfilt(B, A, data_section[n][data_section[n]<18])
#     plt.plot(distance_km[data_section[n]<18], data_f, label = 'section '+str(n), color = ccc[n])


    
# plt.title("Butterworth filter, window size "+str(window_size_km)+' km')
# plt.xlabel('distance, km')
# plt.ylabel('wind speed, m/s')